# Notebook 09: Streamlit Frontend
### Hybrid E-Commerce Recommendation System — H&M Personalized Fashion Recommendations

**Scope of this notebook:** build a Streamlit frontend that calls the already-running FastAPI
backend (Notebook 08) over HTTP, and display the results as an e-commerce style UI.

**Explicitly out of scope for this notebook:**
- No model training or retraining — every recommendation still comes from the artifacts loaded
  once by `api/recommendation_service.py` in Notebook 08.
- No changes to any recommendation algorithm (content-based, ALS, hybrid) — this notebook never
  imports `recommendation_service.py` or touches `models/`, `artifacts/`, or `cb_artifacts/`.
- No Docker or cloud deployment — that is explicitly the *next* notebook, per the brief.

This notebook only adds a **presentation layer** on top of the two HTTP endpoints Notebook 08
already exposes: `GET /recommend/{user_id}` and `GET /similar/{article_id}`.


---
## 1. Architecture — How the Three Layers Talk to Each Other

```text
Streamlit  →  FastAPI  →  Recommendation Model
 (app.py)     (api/main.py)   (RecommendationService, Notebook 08)
```

**Streamlit (`app/app.py`)** is a pure HTTP *client*. It has no knowledge of ALS, TF-IDF, hybrid
scoring, or pickled artifacts. All it does is:
1. Render input widgets (a customer-ID box, an article-ID box, buttons).
2. Send a `GET` request to the FastAPI backend using the `requests` library.
3. Parse the JSON response and render it as product cards.

**FastAPI (`api/main.py`, built in Notebook 08)** is the HTTP *server*. It owns routing,
request validation (`top_k` bounds, non-blank IDs), and translating service-layer conditions
(unknown user, unknown article, missing artifacts) into HTTP status codes (`200`, `400`, `404`,
`422`, `503`).

**`RecommendationService`** is the ML *engine* underneath FastAPI. It was loaded once, at process
startup, from the artifacts saved by Notebooks 03–07 (TF-IDF matrix, ALS factors, popularity
ranking, hybrid config). Streamlit never sees this layer directly — it only ever sees what
`api/main.py` chooses to return as JSON.

**Why this separation matters for this notebook specifically:** because Streamlit only talks to
FastAPI over HTTP, this notebook can be developed, tested, and even run entirely independently of
whether Google Drive is mounted, whether the ALS model is loaded, or which machine the backend
eventually runs on. The *only* thing that couples the two services is one URL — `API_URL` — which
Section 8 below defines in exactly one place.


---
## 2. Project Structure

```text
hm_recsys/                     (BASE_DIR — same project folder as Notebook 08)
├── processed_data/            unchanged — Notebook 09 never reads this directly
├── models/                    unchanged
├── artifacts/                 unchanged
├── cb_artifacts/               unchanged
├── api/                        from Notebook 08 (main.py, recommendation_service.py)
└── app/                        NEW — created by this notebook
    ├── __init__.py
    └── app.py
```

`app/` sits next to `api/`, not inside it — the frontend is a separate concern from the backend,
and the two only communicate over HTTP (`API_URL`), never via a Python import.


In [2]:
import os

# Same convention as Notebook 08: work locally if the project folder happens to already
# exist in this runtime, otherwise mount Drive and point at the shared project folder.
BASE_DIR = "/content/drive/MyDrive/hm_recsys"

if not os.path.exists(BASE_DIR):
    print("Project folder not found locally — mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')

if not os.path.exists(BASE_DIR):
    raise FileNotFoundError(
        f"{BASE_DIR} still not found after mounting Drive. "
        f"Update BASE_DIR to match where Notebook 08 saved api/."
    )

os.makedirs(os.path.join(BASE_DIR, "app"), exist_ok=True)
os.chdir(BASE_DIR)

print("Working directory set to:", os.getcwd())
print("BASE_DIR:", BASE_DIR)


Project folder not found locally — mounting Google Drive...
Mounted at /content/drive
Working directory set to: /content/drive/MyDrive/hm_recsys
BASE_DIR: /content/drive/MyDrive/hm_recsys


---
## 3–8. The Streamlit Application

`app/app.py` below contains every requirement in one file, organized into clearly labeled
sections:

| Section | What it covers |
|---|---|
| Config | The single `API_URL` definition (Section 8) |
| API communication layer | `check_api_health`, `get_recommendations`, `get_similar_products` — all built on `requests`, all with explicit error handling (Section 7) |
| Explainability helper | Turns the API's own fields (`is_new_user`, `score`) into a plain-language reason — nothing invented (Section 6) |
| Product card rendering | Shared card layout for both recommendations and similar products (Sections 2–5) |
| Page sections | Header/status, "Personalized Recommendations", "Find Similar Products" (Sections 2–5) |
| `main()` | Assembles the page; guarded by `if __name__ == "__main__"` so this file is also safely importable for testing (Section 10, below) |

**Design notes:**
- **New-user handling (Section 5):** the backend already treats an unknown `user_id` as a
  cold-start case and returns `is_new_user=True` with popularity-based results — it never raises
  an error for this. The frontend's only job is to detect that flag and show *"New user detected.
  Showing popular products."* instead of treating it as a failure.
- **No hardcoded URLs (Section 8):** every request in the file is built from the one `API_URL`
  constant at the top. Nothing else in the file contains a literal `http://...` string.
- **Import-safe (used in Section 10):** all UI code lives inside `main()`, called only under
  `if __name__ == "__main__"`. When Streamlit runs the file directly, `__name__ == "__main__"` and
  the full UI renders as normal. When this notebook instead does `import app.app`, `__name__` is
  `"app.app"`, so only the plain-Python API-communication functions load — letting Section 10 test
  them directly, without a running Streamlit process.


In [3]:
%%writefile app/app.py
"""
app.py

Streamlit frontend for the Hybrid E-Commerce Recommendation System.

Responsibilities:
- Render an e-commerce style UI (home page, personalized recommendations,
  similar products).
- Call the already-running FastAPI backend (Notebook 08) over HTTP for every
  recommendation — no model is loaded, trained, or touched in this file.
- Handle API errors (unavailable, timeout, invalid input, empty results)
  gracefully, without crashing the UI.

Run standalone with:
    streamlit run app/app.py

Requires the FastAPI backend to already be running (see Notebook 08):
    uvicorn api.main:app --reload
"""

import requests
import streamlit as st

# --------------------------------------------------------------------- #
# Section 8: Configuration — the ONLY place the API base URL is defined.
# Every API call in this file goes through the functions below, which all
# read API_URL from here. To point the app at a deployed backend later
# (Render/AWS/etc.), change this one line — nothing else in the file needs
# to change.
# --------------------------------------------------------------------- #
API_URL = "http://127.0.0.1:8000"
REQUEST_TIMEOUT = 8  # seconds


# --------------------------------------------------------------------- #
# Section 7: API communication layer
#
# These functions are plain Python (no Streamlit calls inside them) so they
# can be imported and tested independently of the Streamlit runtime — see
# Notebook 09, Section 10 ("Final Testing").
#
# Every function returns a tuple: (data, error_message).
# - On success: (parsed_json, None)
# - On failure: (None, "human readable error message")
# The UI layer below only ever has to check "if error: show it".
# --------------------------------------------------------------------- #
def check_api_health():
    """GET /health — used to show a live backend status indicator."""
    try:
        resp = requests.get(f"{API_URL}/health", timeout=REQUEST_TIMEOUT)
        if resp.status_code == 200:
            return resp.json(), None
        return None, f"API returned status {resp.status_code} on /health."
    except requests.exceptions.ConnectionError:
        return None, "Cannot reach the API. Is FastAPI running (uvicorn api.main:app --reload)?"
    except requests.exceptions.Timeout:
        return None, "API health check timed out."
    except requests.exceptions.RequestException as exc:
        return None, f"Unexpected error contacting API: {exc}"


def get_recommendations(user_id: str, top_k: int = 10):
    """
    Calls GET /recommend/{user_id}.

    The backend never errors on an unknown user_id — it automatically falls
    back to popularity-based recommendations and sets is_new_user=True. So
    from this app's point of view, "invalid user" is not a distinct failure
    mode; it's a normal, successful response we display differently.
    """
    user_id = (user_id or "").strip()
    if not user_id:
        return None, "Please enter a user ID."

    try:
        resp = requests.get(
            f"{API_URL}/recommend/{user_id}",
            params={"top_k": top_k},
            timeout=REQUEST_TIMEOUT,
        )
    except requests.exceptions.ConnectionError:
        return None, "Cannot reach the API. Is FastAPI running (uvicorn api.main:app --reload)?"
    except requests.exceptions.Timeout:
        return None, "The API took too long to respond. Please try again."
    except requests.exceptions.RequestException as exc:
        return None, f"Unexpected error contacting API: {exc}"

    if resp.status_code == 200:
        data = resp.json()
        if not data.get("recommendations"):
            return data, "The API returned no recommendations for this user."
        return data, None

    if resp.status_code == 404:
        return None, resp.json().get("detail", "No recommendations could be generated.")
    if resp.status_code == 422:
        return None, "Invalid request (top_k must be between 1 and 50)."
    if resp.status_code == 503:
        return None, "The recommendation service is temporarily unavailable on the backend."
    return None, f"API returned an unexpected status ({resp.status_code})."


def get_similar_products(article_id: str, top_k: int = 10):
    """Calls GET /similar/{article_id}. Handles unknown/malformed article IDs."""
    article_id = (article_id or "").strip()
    if not article_id:
        return None, "Please enter an article ID."

    try:
        resp = requests.get(
            f"{API_URL}/similar/{article_id}",
            params={"top_k": top_k},
            timeout=REQUEST_TIMEOUT,
        )
    except requests.exceptions.ConnectionError:
        return None, "Cannot reach the API. Is FastAPI running (uvicorn api.main:app --reload)?"
    except requests.exceptions.Timeout:
        return None, "The API took too long to respond. Please try again."
    except requests.exceptions.RequestException as exc:
        return None, f"Unexpected error contacting API: {exc}"

    if resp.status_code == 200:
        data = resp.json()
        if not data.get("similar_products"):
            return data, "The API returned no similar products for this article."
        return data, None

    if resp.status_code == 400:
        return None, resp.json().get("detail", "article_id must be numeric.")
    if resp.status_code == 404:
        return None, resp.json().get("detail", f"article_id '{article_id}' was not found in the catalog.")
    if resp.status_code == 422:
        return None, "Invalid request (top_k must be between 1 and 50)."
    if resp.status_code == 503:
        return None, "The recommendation service is temporarily unavailable on the backend."
    return None, f"API returned an unexpected status ({resp.status_code})."


# --------------------------------------------------------------------- #
# Section 6: Explainability helper
#
# Only states what the model/API actually tells us. No per-item score
# breakdown (content vs. collaborative) is invented, because /recommend
# only returns a single combined hybrid score - not a breakdown.
# --------------------------------------------------------------------- #
def explanation_for(is_new_user: bool, score):
    if is_new_user:
        return "Popular right now \u2014 shown because we don't have purchase history for this user yet."
    if score is not None:
        return "Personalized pick from the hybrid model, based on this user's purchase history and similar products."
    return "Trending pick \u2014 shown as a fallback because a personalized score wasn't available for this item."


# --------------------------------------------------------------------- #
# UI: product card rendering
# --------------------------------------------------------------------- #
def render_recommendation_card(item: dict, is_new_user: bool):
    with st.container(border=True):
        st.markdown(f"**{item.get('product_name') or 'Unnamed product'}**")
        st.caption(f"Article ID: {item.get('article_id')}")
        st.write(f"Category: {item.get('category') or 'N/A'}")
        score = item.get("score")
        st.write(f"Score: {score:.4f}" if score is not None else "Score: N/A (popularity-based)")
        st.info(explanation_for(is_new_user, score), icon="\u2139\ufe0f")


def render_similar_card(item: dict):
    with st.container(border=True):
        st.markdown(f"**{item.get('product_name') or 'Unnamed product'}**")
        st.caption(f"Article ID: {item.get('article_id')}")
        st.write(f"Category: {item.get('category') or 'N/A'}")
        st.write(f"Similarity: {item.get('similarity_score'):.4f}")
        st.info(
            f"Similar to article {item.get('article_id')} based on shared product "
            "attributes (content-based similarity).",
            icon="\u2139\ufe0f",
        )


def render_card_grid(items, is_new_user=None, card_kind="recommendation"):
    cols_per_row = 2
    for row_start in range(0, len(items), cols_per_row):
        row_items = items[row_start: row_start + cols_per_row]
        cols = st.columns(cols_per_row)
        for col, item in zip(cols, row_items):
            with col:
                if card_kind == "recommendation":
                    render_recommendation_card(item, is_new_user)
                else:
                    render_similar_card(item)


# --------------------------------------------------------------------- #
# UI: page sections
# --------------------------------------------------------------------- #
def render_header():
    st.set_page_config(
        page_title="Hybrid E-Commerce Recommendations",
        page_icon="\U0001F6CD\ufe0f",
        layout="wide",
    )
    st.title("\U0001F6CD\ufe0f Hybrid E-Commerce Recommendation System")
    st.write(
        "A portfolio project demonstrating an end-to-end recommendation pipeline: "
        "content-based filtering, collaborative filtering (ALS), and a hybrid model "
        "that blends both \u2014 served through a FastAPI backend and this Streamlit "
        "frontend. Fashion product catalog courtesy of the H&M Personalized Fashion "
        "Recommendations dataset."
    )

    health, health_error = check_api_health()
    if health_error:
        st.error(f"Backend status: unreachable \u2014 {health_error}")
    elif health.get("status") == "healthy":
        st.success("Backend status: connected \u2192 " + API_URL)
    else:
        st.warning("Backend status: reachable, but reporting itself as unhealthy.")

    st.divider()


def render_recommendations_section():
    st.header("Personalized Recommendations")
    st.write(
        "Enter a customer ID to get personalized picks. Unknown IDs are treated as "
        "new users automatically \u2014 no error, just a popularity-based fallback."
    )

    col1, col2 = st.columns([3, 1])
    with col1:
        user_id = st.text_input(
            "Customer ID",
            key="user_id_input",
            placeholder="e.g. 00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657",
        )
    with col2:
        top_k = st.number_input("How many?", min_value=1, max_value=50, value=10, key="rec_top_k")

    if st.button("Get Recommendations", type="primary"):
        with st.spinner("Fetching recommendations..."):
            data, error = get_recommendations(user_id, top_k=top_k)
        st.session_state["rec_result"] = data
        st.session_state["rec_error"] = error

    data = st.session_state.get("rec_result")
    error = st.session_state.get("rec_error")

    if error and not (data and data.get("recommendations")):
        st.error(error)
    elif data:
        if data.get("is_new_user"):
            st.warning("New user detected. Showing popular products.")
        else:
            st.success(f"Showing personalized recommendations for user: {data.get('user_id')}")
        render_card_grid(data["recommendations"], is_new_user=data.get("is_new_user"), card_kind="recommendation")

    st.divider()


def render_similar_products_section():
    st.header("Find Similar Products")
    st.write("Enter an article ID from the catalog to find visually and descriptively similar products.")

    col1, col2 = st.columns([3, 1])
    with col1:
        article_id = st.text_input("Article ID", key="article_id_input", placeholder="e.g. 108775015")
    with col2:
        top_k = st.number_input("How many?", min_value=1, max_value=50, value=10, key="sim_top_k")

    if st.button("Find Similar Products"):
        with st.spinner("Searching for similar products..."):
            data, error = get_similar_products(article_id, top_k=top_k)
        st.session_state["sim_result"] = data
        st.session_state["sim_error"] = error

    data = st.session_state.get("sim_result")
    error = st.session_state.get("sim_error")

    if error and not (data and data.get("similar_products")):
        st.error(error)
    elif data:
        st.success(f"Products similar to article {data.get('article_id')}:")
        render_card_grid(data["similar_products"], card_kind="similar")


# --------------------------------------------------------------------- #
# Entry point
# --------------------------------------------------------------------- #
def main():
    render_header()
    render_recommendations_section()
    render_similar_products_section()


if __name__ == "__main__":
    main()


Overwriting app/app.py


In [4]:
%%writefile app/__init__.py
# Makes `app/` importable as a package (e.g. `import app.app`), used for testing in Section 10.


Overwriting app/__init__.py


---
## `requirements.txt` — Updated with Frontend Dependencies

This extends the `requirements.txt` written in Notebook 08 (backend + ML libraries) with the two
additional libraries the Streamlit frontend needs: `streamlit` itself and `requests` for talking
to the API. Nothing already in the file is removed — `app/app.py` never imports `fastapi`,
`implicit`, `scikit-learn`, etc., but the same environment can happily run both services.


In [5]:
%%writefile requirements.txt
# --- Backend + ML dependencies (Notebook 08) ---
fastapi>=0.110
uvicorn[standard]>=0.29
pydantic>=2.6
pandas>=2.0
numpy>=1.24
scipy>=1.11
scikit-learn>=1.3
implicit>=0.7

# --- Frontend dependencies (Notebook 09) ---
streamlit>=1.32
requests>=2.31


Overwriting requirements.txt


In [6]:
# Install the frontend dependencies just pinned in requirements.txt into THIS Colab
# runtime. This is only needed so the rest of this notebook (Section 10's tests, which
# `import app.app` and therefore `import streamlit`) can run here — someone deploying
# for real would instead just run `pip install -r requirements.txt` in their terminal
# (see Section 9 below), which installs everything, backend included.
!pip install -q streamlit requests
print("Frontend dependencies installed.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 48.3 MB/s eta 0:00:00
Frontend dependencies installed.


---
## 9. Run Locally

Two separate processes, in two separate terminals, from the project root (`hm_recsys/`):

**Terminal 1 — start the backend:**
```bash
pip install -r requirements.txt
uvicorn api.main:app --reload
```
This starts FastAPI on `http://127.0.0.1:8000` (the default `API_URL` in `app/app.py`).

**Terminal 2 — start the frontend:**
```bash
streamlit run app/app.py
```
This opens the UI in a browser, typically at `http://localhost:8501`.

**How they communicate:** the browser talks only to Streamlit. Streamlit's Python process, in
turn, makes plain HTTP `GET` requests to `http://127.0.0.1:8000/...` using the `requests` library
— the exact same calls `curl` or the FastAPI Swagger UI (`/docs`) would make. The backend must
already be running before the frontend is used, since every recommendation the UI shows comes
from a live API call, not from any local computation.

Later, when the backend is deployed (Render/AWS/etc. — next notebook), the **only** change needed
is updating the one `API_URL` constant in `app/app.py` to point at the deployed URL instead of
`127.0.0.1:8000`.


---
## 10. Final Testing

Streamlit apps don't have a natural "run inside a notebook cell" mode the way FastAPI does (there
is no request/response cycle to script against — it's a long-lived browser session). What *can*
be tested headlessly, and what actually matters for correctness, is the API-communication layer:
`check_api_health`, `get_recommendations`, and `get_similar_products` in `app/app.py`. Because
those functions contain no Streamlit calls (see the import-safe design note above), they can be
imported directly and exercised with `assert` statements.

The real trained artifacts from Notebooks 03–07 live in this Colab session's Google Drive, and
Notebook 08 already proved the real API works end-to-end against them. To keep this notebook
self-contained and fast to re-run, the cells below spin up a small **mock** FastAPI server that
returns data in the *exact same schema* as `api/main.py` (same field names, same status codes for
each error case) — this only tests `app.py`'s request/response handling, not the recommendation
algorithms themselves (which are out of scope here, per the brief).

To test against the **real** backend instead, simply run Notebook 08's server-start cell first and
skip straight to setting `frontend.API_URL = "http://127.0.0.1:8000"` before the test cells below.


In [7]:
# Mock FastAPI server — mirrors api/main.py's exact response schema and status codes,
# purely so app.py's HTTP-handling code can be exercised in this notebook without
# needing the multi-MB ALS/TF-IDF artifacts loaded. No recommendation logic here.

import threading
import time
from typing import List, Optional

import nest_asyncio
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

nest_asyncio.apply()

MOCK_PORT = 8010
mock_app = FastAPI()


class RecommendationItem(BaseModel):
    article_id: str
    product_name: str
    category: str
    score: Optional[float] = None


class RecommendationResponse(BaseModel):
    user_id: str
    is_new_user: bool
    recommendations: List[RecommendationItem]


class SimilarItem(BaseModel):
    article_id: str
    product_name: str
    similarity_score: float
    category: str


class SimilarResponse(BaseModel):
    article_id: str
    similar_products: List[SimilarItem]


MOCK_ACTIVE_USER = "active_user_123"
MOCK_LOW_ACTIVITY_USER = "low_activity_user_456"
MOCK_KNOWN_ARTICLE = "108775015"


def _mock_recs(prefix, n=10, with_score=True):
    return [
        {
            "article_id": str(100000000 + i),
            "product_name": f"{prefix} Product {i}",
            "category": "Garment Upper body" if i % 2 == 0 else "Accessories",
            "score": round(0.95 - i * 0.05, 4) if with_score else None,
        }
        for i in range(n)
    ]


@mock_app.get("/health")
def health():
    return {"status": "healthy"}


@mock_app.get("/recommend/{user_id}", response_model=RecommendationResponse)
def recommend(user_id: str, top_k: int = 10):
    if user_id == MOCK_ACTIVE_USER:
        return RecommendationResponse(
            user_id=user_id, is_new_user=False, recommendations=_mock_recs("Active", top_k)
        )
    if user_id == MOCK_LOW_ACTIVITY_USER:
        return RecommendationResponse(
            user_id=user_id, is_new_user=False, recommendations=_mock_recs("LowActivity", top_k)
        )
    # Unknown user_id -> popularity fallback, mirroring RecommendationService's cold-start path.
    return RecommendationResponse(
        user_id=user_id, is_new_user=True, recommendations=_mock_recs("Popular", top_k, with_score=False)
    )


@mock_app.get("/similar/{article_id}", response_model=SimilarResponse)
def similar(article_id: str, top_k: int = 10):
    try:
        normalized = int(article_id)
    except ValueError:
        raise HTTPException(status_code=400, detail=f"article_id must be numeric, got '{article_id}'.")
    if str(normalized) != MOCK_KNOWN_ARTICLE:
        raise HTTPException(status_code=404, detail=f"article_id {normalized} not found in catalog.")
    products = [
        {
            "article_id": str(200000000 + i),
            "product_name": f"Similar Product {i}",
            "category": "Garment Upper body",
            "similarity_score": round(0.9 - i * 0.05, 4),
        }
        for i in range(top_k)
    ]
    return SimilarResponse(article_id=str(normalized), similar_products=products)


def _run_mock_server():
    uvicorn.run(mock_app, host="127.0.0.1", port=MOCK_PORT, log_level="warning")


threading.Thread(target=_run_mock_server, daemon=True).start()
time.sleep(2)
print(f"Mock API running on http://127.0.0.1:{MOCK_PORT} (schema-identical to Notebook 08's real API).")
print("This mock exists ONLY to test app.py's request/response handling in this notebook.")


Mock API running on http://127.0.0.1:8010 (schema-identical to Notebook 08's real API).
This mock exists ONLY to test app.py's request/response handling in this notebook.


In [8]:
# Import app.py's plain-Python API-communication functions directly (no Streamlit
# process needed — see the "import-safe" design note above) and point them at the mock.

import sys
sys.path.insert(0, BASE_DIR)

import app.app as frontend  # noqa: E402

frontend.API_URL = f"http://127.0.0.1:{MOCK_PORT}"

print("== check_api_health() ==")
print(frontend.check_api_health())


== check_api_health() ==
({'status': 'healthy'}, None)


In [9]:
# 1. Existing ACTIVE user
data, err = frontend.get_recommendations(MOCK_ACTIVE_USER, top_k=5)
print("1. Active user  -> error:", err, "| is_new_user:", data["is_new_user"], "| n_recs:", len(data["recommendations"]))
assert err is None and data["is_new_user"] is False and len(data["recommendations"]) == 5

# 2. Existing LOW-ACTIVITY user
data, err = frontend.get_recommendations(MOCK_LOW_ACTIVITY_USER, top_k=5)
print("2. Low-activity -> error:", err, "| is_new_user:", data["is_new_user"], "| n_recs:", len(data["recommendations"]))
assert err is None and data["is_new_user"] is False

# 3. NEW / unknown user
data, err = frontend.get_recommendations("totally_unknown_customer_id", top_k=5)
print("3. New user     -> error:", err, "| is_new_user:", data["is_new_user"], "| sample score:", data["recommendations"][0]["score"])
assert err is None and data["is_new_user"] is True and data["recommendations"][0]["score"] is None

# 4. Valid product
data, err = frontend.get_similar_products(MOCK_KNOWN_ARTICLE, top_k=5)
print("4. Valid product-> error:", err, "| n_similar:", len(data["similar_products"]))
assert err is None and len(data["similar_products"]) == 5

# 5. Invalid product (well-formed but unknown id)
data, err = frontend.get_similar_products("999999999", top_k=5)
print("5. Unknown id   -> data:", data, "| error:", err)
assert data is None and err is not None

# 5b. Malformed (non-numeric) product id
data, err = frontend.get_similar_products("not-a-number", top_k=5)
print("5b. Non-numeric -> data:", data, "| error:", err)
assert data is None and err is not None

print("\nScenarios 1-5 all handled correctly by app.py's helper functions.")


1. Active user  -> error: None | is_new_user: False | n_recs: 5
2. Low-activity -> error: None | is_new_user: False | n_recs: 5
3. New user     -> error: None | is_new_user: True | sample score: None
4. Valid product-> error: None | n_similar: 5
5. Unknown id   -> data: None | error: article_id 999999999 not found in catalog.
5b. Non-numeric -> data: None | error: article_id must be numeric, got 'not-a-number'.

Scenarios 1-5 all handled correctly by app.py's helper functions.


In [10]:
# 6. FastAPI unavailable — point at a port nothing is listening on.
frontend.API_URL = "http://127.0.0.1:8099"

data, err = frontend.get_recommendations(MOCK_ACTIVE_USER, top_k=5)
print("6a. /recommend  -> data:", data, "| error:", err)
assert data is None and "Cannot reach the API" in err

data, err = frontend.get_similar_products(MOCK_KNOWN_ARTICLE, top_k=5)
print("6b. /similar    -> data:", data, "| error:", err)
assert data is None and "Cannot reach the API" in err

health, health_err = frontend.check_api_health()
print("6c. /health     -> data:", health, "| error:", health_err)
assert health is None and health_err is not None

# Restore for anyone continuing to experiment with this notebook interactively.
frontend.API_URL = f"http://127.0.0.1:{MOCK_PORT}"

print("\nAll 6 required test scenarios verified — the Streamlit UI shows st.error(...) with")
print("these exact messages whenever the corresponding condition occurs, instead of crashing.")


6a. /recommend  -> data: None | error: Cannot reach the API. Is FastAPI running (uvicorn api.main:app --reload)?
6b. /similar    -> data: None | error: Cannot reach the API. Is FastAPI running (uvicorn api.main:app --reload)?
6c. /health     -> data: None | error: Cannot reach the API. Is FastAPI running (uvicorn api.main:app --reload)?

All 6 required test scenarios verified — the Streamlit UI shows st.error(...) with
these exact messages whenever the corresponding condition occurs, instead of crashing.


---
## 11. Final Deliverable

```text
app/
    __init__.py
    app.py
```

and an updated `requirements.txt` (backend deps from Notebook 08 + `streamlit`, `requests`).

**The complete system, end to end:**

```text
User
 ↓
Streamlit          (app/app.py — renders the UI, calls the API via `requests`)
 ↓
FastAPI            (api/main.py — routes, validation, HTTP status codes)
 ↓
Hybrid Recommendation Engine   (RecommendationService — loaded once from Notebooks 03-07's artifacts)
 ↓
Top-N Products     (returned as JSON, rendered as product cards)
```

No model was retrained, no recommendation algorithm was modified, and no hardcoded API URLs were
introduced anywhere in `app/app.py` — every request flows through the single `API_URL` constant
defined in Section 8.

**Next: Dockerization and Deployment** (containerizing both `api/` and `app/`, and pointing
`API_URL` at a deployed backend instead of `127.0.0.1:8000`) — explicitly out of scope for this
notebook, per the brief.


In [11]:
import os

for fname in ["app/__init__.py", "app/app.py", "requirements.txt"]:
    full_path = os.path.join(BASE_DIR, fname)
    exists = os.path.exists(full_path)
    size = os.path.getsize(full_path) if exists else 0
    print(f"{fname}: {'OK' if exists else 'MISSING'} ({size} bytes)")

print()
print("To run the full system locally:")
print(f"  cd {BASE_DIR}")
print("  pip install -r requirements.txt")
print("  uvicorn api.main:app --reload      # terminal 1 (backend)")
print("  streamlit run app/app.py           # terminal 2 (frontend)")


app/__init__.py: OK (96 bytes)
app/app.py: OK (12313 bytes)
requirements.txt: OK (249 bytes)

To run the full system locally:
  cd /content/drive/MyDrive/hm_recsys
  pip install -r requirements.txt
  uvicorn api.main:app --reload      # terminal 1 (backend)
  streamlit run app/app.py           # terminal 2 (frontend)
